In [ ]:
import pypsa
import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import seaborn as sns
import re
from _helpers import load_config

### Hypothesis

`n.statistics.energy-balance()` shows the aggregation of all single generators, that is why it is never 0
How it should be: single quality classes should get down to 0, but then the question arises: should you throw them into one model, or separate? How is the local demand handled? How does this align with the philosophy of multiple demands?

### Paths

In [ ]:
product = "steel"  # "eaf" or "steel" or "eaf-grid"
region = "Europe"  # e.g. "Europe"

In [ ]:
single_region = f"../../resources/lco-{product}/cost_year~2030/{region}/network_1-0partload.nc"
# single_region = f"../../resources/lcos/cost_year~2030/Europe/network_1.nc"

### Read product model

In [ ]:
n = pypsa.Network(single_region)

In [ ]:
n.statistics()

### RE profiles

In [ ]:
# Generate a df with p_nom_opt, p_nom_max and flh
gen_df = n.generators[["p_nom_opt", "p_nom_max"]].copy()
# remove iron ore DRI-ready (exp)
gen_df = gen_df[~gen_df.index.str.contains("iron ore DRI-ready")]
gen_df["flh"] = n.generators_t.p_max_pu.sum(axis=0) 
gen_df = gen_df.rename(columns={"p_nom_opt": "installed (MW)", "p_nom_max": "potential (MW)", "flh": "flh"})
# reorder columns
gen_df = gen_df[["flh", "potential (MW)", "installed (MW)"]]
gen_df = gen_df.sort_values(by="flh", ascending=False)
sum_row = pd.DataFrame(gen_df.sum(numeric_only=True)).T
sum_row.index = ["Total"]
gen_df["utilization"] = gen_df["installed (MW)"] / gen_df["potential (MW)"]
gen_df = pd.concat([gen_df, sum_row])
gen_df.round(1)

In [ ]:
n.generators_t.p_max_pu.loc[:, n.generators[n.generators.carrier == "wind"].index].mean(axis=1).plot(figsize=(15, 5), grid=True, ylim=(0, 1))

In [ ]:
n.generators_t.p_max_pu.loc[:, "onwind 1"].plot(figsize=(15, 5), grid=True, ylim=(0, 1))

In [ ]:
n.generators_t.p_max_pu.loc[:, "onwind 11"].plot(figsize=(15, 5), grid=True, ylim=(0, 1))

### Actual feed in

In [ ]:
n.generators_t.p.loc[:, n.generators[n.generators.carrier=="wind"].index].mean().round(1)

In [ ]:
n.generators_t.p.loc[:, n.generators[n.generators.carrier=="wind"].index].plot(figsize=(15, 5), grid=True)

In [ ]:
n.statistics.energy_balance.iplot.area(bus_carrier="electricity")